# E791 generation and fit with Laura++-style integration

This notebook performs an end-to-end closure test for the seven-component E791 $D^+\to\pi^-\pi^+\pi^+$ Fit 2 model. Pseudo-data are generated from the known complex coefficients, and the twelve free Cartesian coefficient parameters are fitted with the Laura++-style Gauss--Legendre normalization. The $\rho(770)$ coefficient is fixed to $1+0i$ as the reference amplitude.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant, Parameter,
    RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()

## E791 Fit 2 truth model

In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
fit2_polar = {
    "sigma": (1.17, 205.7), "rho770": (1.00, 0.0),
    "NR": (0.48, 57.3), "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3), "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def internal_xy(name):
    magnitude, phase_deg = fit2_polar[name]
    if name == "NR":
        phase_deg += 180.0
    phase = np.deg2rad(phase_deg)
    return magnitude * np.cos(phase), magnitude * np.sin(phase)

truth = {}

def free_coefficient(name):
    x_truth, y_truth = internal_xy(name)
    truth[f"{name}.x"] = float(x_truth)
    truth[f"{name}.y"] = float(y_truth)
    return RealImag(
        Parameter.coefficient(f"{name}.x", x_truth, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", y_truth, owner=name, step=0.01),
    )

coefficients = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}

components = [
    Resonance("sigma", (0, 1), coefficients["sigma"], mass=0.478, width=0.324, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", (0, 1), coefficients["rho770"], mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", (0, 1), coefficients["f0_980"], mass=0.975, width=0.044, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0, 1), coefficients["f2_1270"], mass=1.275, width=0.185, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0, 1), coefficients["f0_1370"], mass=1.434, width=0.173, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0, 1), coefficients["rho1450"], mass=1.465, width=0.310, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]

model = DecayModel(
    channel, components,
    normalize_components=True,
    normalization_method="laura",
    normalization_bin_width=0.005,
)
print(f"free parameters: {len(model.parameters)}")
print(f"Laura normalization points: {model.normalization_sample.size:,}")

## Generate pseudo-data

The phase-space pool carries its proposal weights. Resampling uses $w_{PS}|A(\Phi;\theta_{true})|^2$. The same Laura++ normalization convention is used later in the fit.

In [ ]:
N_POOL = 300_000
N_DATA = 30_000
pool = model.generate_phase_space(N_POOL, seed=2000)
pool_cache = model.prepare_cache(pool)
truth_intensity, truth_normalization = pool_cache.evaluate(truth)
target_weights = pool.weights * truth_intensity
data = weighted_resample(
    jax.random.key(791), pool, target_weights, N_DATA, replace=True
)
print(f"pool events: {pool.size:,}")
print(f"toy events : {data.size:,}")
print(f"truth normalization: {float(truth_normalization):.12g}")
assert bool(jnp.all(jnp.isfinite(target_weights)))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
hist = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=100)
fig.colorbar(hist[3], ax=ax, label="events")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("E791 Fit 2 pseudo-data")
plt.show()

## Cached unbinned NLL and deterministic start

In [ ]:
cache = model.prepare_cache(data)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    log_intensity = jnp.log(jnp.clip(intensity, min=1.0e-300))
    return -jnp.sum(log_intensity) + data.size * jnp.log(normalization)

rng = np.random.default_rng(314159)
start_values = {
    parameter.name: float(truth[parameter.name] + rng.normal(0.0, 0.20))
    for parameter in model.parameters if not parameter.fixed
}
minimizer = Minimizer(nll, model.parameters, tolerance=1.0e-4, verbose=1)
print(f"NLL(truth): {float(nll(truth)):.6f}")
print(f"NLL(start): {float(nll(start_values)):.6f}")

In [ ]:
gradient_check = minimizer.check_gradient(
    start_values, step_scale=1.0e-5, print_table=True
)

## Fit and closure table

In [ ]:
result = minimizer.fit(
    start_values=start_values, simplex=False, ncall=100_000
)
fit_values = {
    parameter.name: float(result.values[parameter.name])
    for parameter in model.parameters if not parameter.fixed
}
print(f"valid          : {bool(result.valid)}")
print(f"NLL(truth)     : {float(nll(truth)):.6f}")
print(f"NLL(fit)       : {float(result.fval):.6f}")
print(f"fit-truth NLL  : {float(result.fval - nll(truth)):.6f}")
print(f"EDM            : {float(result.fmin.edm):.3e}")
print(f"function calls : {int(result.nfcn)}")
assert bool(result.valid)

In [ ]:
rows = []
print(f"{'parameter':16s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for parameter in model.parameters:
    if parameter.fixed:
        continue
    name = parameter.name
    truth_value = float(truth[name])
    start = float(start_values[name])
    fitted = float(result.values[name])
    error = float(result.errors[name])
    pull = (fitted - truth_value) / error
    rows.append((name, truth_value, start, fitted, error, pull))
    print(f"{name:16s} {truth_value:10.5f} {start:10.5f} {fitted:10.5f} {error:10.5f} {pull:9.3f}")

In [ ]:
labels = [row[0] for row in rows]
pulls = np.asarray([row[5] for row in rows])
fig, ax = plt.subplots(figsize=(10, 5))
ax.axhspan(-2.0, 2.0, alpha=0.12, color="tab:green")
ax.axhline(0.0, color="black", linewidth=1)
ax.scatter(np.arange(len(labels)), pulls)
ax.set_xticks(np.arange(len(labels)), labels, rotation=45, ha="right")
ax.set_ylabel(r"pull $(\hat\theta-\theta_{true})/\sigma$")
ax.set_title("E791 coefficient closure with Laura++ integration")
plt.tight_layout()
plt.show()

## Projection check

In [ ]:
def projection(values, bins):
    intensity, _ = pool_cache.evaluate(values)
    weights = np.asarray(pool.weights * intensity)
    h12, _ = np.histogram(np.asarray(pool.s12), bins=bins, weights=weights)
    h13, _ = np.histogram(np.asarray(pool.s13), bins=bins, weights=weights)
    return h12 + h13

data_projection = np.concatenate((np.asarray(data.s12), np.asarray(data.s13)))
bins = np.linspace(data_projection.min(), data_projection.max(), 100)
centres = 0.5 * (bins[:-1] + bins[1:])
observed, _ = np.histogram(data_projection, bins=bins)
expected_truth = projection(truth, bins)
expected_fit = projection(fit_values, bins)
expected_truth *= observed.sum() / expected_truth.sum()
expected_fit *= observed.sum() / expected_fit.sum()

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(centres, observed, yerr=np.sqrt(np.maximum(observed, 1)), fmt=".", label="toy")
ax.step(centres, expected_fit, where="mid", label="fit")
ax.step(centres, expected_truth, where="mid", linestyle="--", label="truth")
ax.set_xlabel(r"$m^2(\pi^-\pi^+)$ [GeV$^2$]")
ax.set_ylabel("entries / bin")
ax.legend()
plt.show()

A single toy does not establish absence of bias. Production validation should repeat this workflow over an ensemble and study the mean and width of every pull distribution while increasing the Laura++ quadrature order.